# Create Blue Planet Prize Awards (PRIZE PATTERN, static-HTML scrape)

Ingest of the **Blue Planet Prize**, given annually since 1992 by the **Asahi Glass Foundation** for outstanding contributions to environmental science and policy. Source: `af-info.or.jp/en/blueplanet/` — the foundation's own English-language site (no auth, no aggregator). Method #5 on the runbook ladder (static-HTML scrape).

**Awarding body:** Asahi Glass Foundation — `F4320309996` (JP, DOI `10.13039/100007684`, no ROR; alternate titles include `公益財団法人 旭硝子財団` and `The Asahi Glass Foundation`).

**Corpus** (verified 2026-05-21 full local scrape): **71 laureate rows** across **34 distinct years** (1992-2025). Two laureates per year is the default; three years (2003, 2012, 2014) honored three recipients in joint-prize fashion. 65 individuals + 6 organizations.

**Three novel ingestion challenges documented in this PR** — these are the reasons the script's parse logic is more involved than e.g. Crafoord or Holberg:
1. **URL format schema drift**: years 1992-2023 use `list-YYYY.html` (hyphen), years 2024+ use `list_YYYY.html` (underscore). The foundation changed convention silently. The script tries hyphen first then falls back to underscore on 404.
2. **UTF-8 mis-declared charset**: the site serves UTF-8 but does NOT declare `charset` on `Content-Type`, so `requests` falls back to ISO-8859-1 by default. The Japanese wave dash `～` (U+FF5E) used in lifespan notation (`1928～2005` for deceased laureates) corrupts into a 3-byte Latin-1 sequence `\xef\xbd\x9e` and breaks lifespan parsing. The script forces `resp.encoding = 'utf-8'` when the default is Latin-1.
3. **Joint-prize pages use a different DOM structure**: 2003 / 2012 / 2014 share a single anchored section between 2-3 laureates, with names in `<div class="c-card__text">` cards inside a `c-card__body` block, rather than the per-section `<h2 class="c-ttl-lv02">` heading used on most years. The script's parser has a fallback that walks the joint-prize card layout when the primary heading parse returns fewer than 2 records.

**Pattern:** PRIZE (single named recipient per category per year). Some recipients are organizations (IIED, IUCN, IPBES, etc.) rather than individuals — the script distinguishes via `recipient_kind` (individual/organization), parsed from the body text's `Born in YYYY` / `Founded in YYYY` markers (or `YYYY～YYYY` lifespan for deceased individuals).

**Amount choice — constant USD 500,000 per laureate:**
The Asahi Glass Foundation's own `/en/blueplanet/about.html` page documents the prize amount as **"500,000 USD in prize money"** per recipient (verified 2026-05-21). Historical amounts were quoted in JPY 50,000,000 per laureate, but the foundation no longer publishes year-banded amounts — they have standardized on the USD figure. We ship the documented current rule uniformly across all years and document the assumption here. `§6.7 amount-coverage NOT waived` — expect 100%.

**Schema:**
- `funder_award_id` = `blue-planet-{year}-{anchor}-{slug}` (e.g., `blue-planet-1992-text-01-syukuro-manabe`). Anchor is the source page's section ID (`text-01`, `text-02`, `text-03`, or synthetic `card-NN` for joint-prize fallback). Verified unique across the 71-row corpus.
- `lead_investigator` is PERSON-LEVEL for individual recipients (given_name + family_name from runbook §2.4.1 split), and **org-level** for the 6 organizational recipients (NULL given/family, affiliation.name = org name). The recipient_kind column captures the distinction.
- `country` from the heading parenthetical (e.g. `(USA)`, `(Sweden)`); affiliation country falls back to `JP` only when recipient is an organization with founding info but no country — see Step 2 CASE.
- `funder_scheme = 'Blue Planet Prize'`. Single program.
- `funding_type = 'prize'` (per runbook prize-pattern convention).
- `declined` always FALSE — Foundation only publishes awarded laureates.

**Source authority** — `af-info.or.jp` is the Asahi Glass Foundation's own site. No aggregator. Method #5 (static-HTML scrape) on the runbook ladder.

**Prerequisites**: Run `scripts/local/blue_planet_prize_to_s3.py` first to scrape 34 year pages (~30 sec wall-clock at 0.4s throttle), parse, write parquet, upload to S3.

**Data source**: `https://www.af-info.or.jp/en/blueplanet/list-YYYY.html` (1992-2023) and `https://www.af-info.or.jp/en/blueplanet/list_YYYY.html` (2024+)

**S3 location**: `s3a://openalex-ingest/awards/blue_planet_prize/blue_planet_prize_laureates.parquet`

## Step 1: Create staging table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.blue_planet_prize_raw
USING delta
AS
SELECT *, current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/blue_planet_prize/blue_planet_prize_laureates.parquet`;

In [ ]:
%sql
SELECT COUNT(*) FROM openalex.awards.blue_planet_prize_raw;

## Step 1.5: Inspect raw + money/currency scan + coverage acknowledgment

All source columns from the static-HTML scrape. **Verified per-row coverage on the full extraction** (71 laureate rows across 34 years, validated 2026-05-21):
- 100% `funder_award_id` (slug+anchor unique by design)
- 100% `name_clean`, `year`
- 100% `amount` = constant $500,000 USD per laureate (foundation's documented ceiling)
- 92% `country` (65/71 — 6 organizations don't have a country code, they have founding location info)
- 92% `affiliation_name` (65/71 — same 6 orgs; for orgs we use the org name itself as affiliation in Step 2)
- 100% `description`
- 83% `birth_year` (59/71 — orgs + a handful of older laureates where Born/lifespan markers couldn't be parsed)
- 34 distinct years
- 65 individuals + 6 organizations

Source columns: `funder_award_id`, `year`, `anchor`, `name_clean`, `raw_heading`, `given_name`, `family_name`, `country`, `recipient_kind`, `birth_year`, `founding_year`, `affiliation_name`, `display_name`, `description`, `amount`, `currency`, `start_date`, `end_date`, `landing_page_url`, `declined`.

In [ ]:
%sql
DESCRIBE openalex.awards.blue_planet_prize_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.blue_planet_prize_raw LIMIT 5;

In [ ]:
%sql
-- §1.5 money-shape scan — amount is the foundation's documented constant
-- $500K USD ceiling. Verify the column shipped correctly with no NULL
-- contamination.
SELECT
    MIN(TRY_CAST(amount AS DOUBLE)) AS min_amount,
    MAX(TRY_CAST(amount AS DOUBLE)) AS max_amount,
    AVG(TRY_CAST(amount AS DOUBLE)) AS avg_amount,
    COUNT(amount) AS non_null,
    COUNT(*) AS total_rows
FROM openalex.awards.blue_planet_prize_raw;

In [ ]:
%sql
-- Year + recipient_kind distribution — confirms 1992-2025 span, 2/year
-- baseline with 3 years (2003/2012/2014) at 3 recipients.
SELECT TRY_CAST(year AS INT) AS year_int,
       recipient_kind,
       COUNT(*) AS rows
FROM openalex.awards.blue_planet_prize_raw
GROUP BY year_int, recipient_kind
ORDER BY year_int, recipient_kind;

## Step 1.6: Fail-fast — verify Asahi Glass Foundation funder row exists

In [ ]:
%sql
-- Must return exactly 1 row. If 0 rows, STOP — the funder is missing
-- from openalex.common.funder. Flag back to the user; do not proceed.
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320309996;

## Step 2: Transform to award schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.blue_planet_prize_awards
USING delta
AS
WITH funder_resolved AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320309996  -- Asahi Glass Foundation
)
SELECT
    abs(xxhash64(CONCAT(
        TRY_CAST(f.funder_id AS STRING), ':', LOWER(n.funder_award_id)
    ))) % 9000000000 AS id,
    n.display_name,
    n.description,
    f.funder_id,
    n.funder_award_id,
    TRY_CAST(n.amount AS DOUBLE) AS amount,
    n.currency,
    struct(
        CONCAT('https://openalex.org/F', TRY_CAST(f.funder_id AS STRING)) AS id,
        f.display_name,
        f.ror_id,
        f.doi
    ) AS funder,
    'prize' AS funding_type,
    'Blue Planet Prize' AS funder_scheme,
    'blue_planet_prize' AS provenance,
    TRY_TO_DATE(n.start_date, 'yyyy-MM-dd') AS start_date,
    TRY_TO_DATE(n.end_date,   'yyyy-MM-dd') AS end_date,
    TRY_CAST(SUBSTRING(n.start_date, 1, 4) AS INT) AS start_year,
    TRY_CAST(SUBSTRING(n.end_date,   1, 4) AS INT) AS end_year,
    -- lead_investigator is PERSON-LEVEL for individuals (given/family
    -- from the script's split_name helper) and ORG-LEVEL for the 6
    -- organizational recipients (NULL person fields; affiliation.name =
    -- the org's own name). recipient_kind drives the branch.
    CASE
        WHEN n.recipient_kind = 'individual' AND n.name_clean IS NOT NULL THEN struct(
            n.given_name AS given_name,
            n.family_name AS family_name,
            CAST(NULL AS STRING) AS orcid,
            TRY_TO_DATE(n.start_date, 'yyyy-MM-dd') AS role_start,
            struct(
                n.affiliation_name AS name,
                CAST(NULL AS STRING) AS country,
                CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
            ) AS affiliation
        )
        WHEN n.recipient_kind = 'organization' AND n.name_clean IS NOT NULL THEN struct(
            CAST(NULL AS STRING) AS given_name,
            CAST(NULL AS STRING) AS family_name,
            CAST(NULL AS STRING) AS orcid,
            TRY_TO_DATE(n.start_date, 'yyyy-MM-dd') AS role_start,
            struct(
                n.name_clean AS name,
                CAST(NULL AS STRING) AS country,
                CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
            ) AS affiliation
        )
        ELSE NULL
    END AS lead_investigator,
    CAST(NULL AS STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >) AS co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >>) AS investigators,
    n.landing_page_url AS landing_page_url,
    CAST(NULL AS STRING) AS doi,
    CONCAT('https://api.openalex.org/works?filter=awards.id:G',
           TRY_CAST(abs(xxhash64(CONCAT(
               TRY_CAST(f.funder_id AS STRING), ':', LOWER(n.funder_award_id)
           ))) % 9000000000 AS STRING)) AS works_api_url,
    n.declined,
    current_timestamp() AS created_date,
    current_timestamp() AS updated_date
FROM openalex.awards.blue_planet_prize_raw n
CROSS JOIN funder_resolved f
WHERE n.funder_award_id IS NOT NULL
  AND n.name_clean IS NOT NULL;

## Step 3: Insert into openalex_awards_raw at priority 91

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'blue_planet_prize' AND priority = 91;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    91 AS priority  -- Blue Planet Prize priority (matches CreateAwards.ipynb registry)
FROM openalex.awards.blue_planet_prize_awards;

## Step 6: Verification

Full §6.1-6.7. Amount-coverage NOT waived (constant $500K per laureate). Verified expectations from the 2026-05-21 extraction:
- Row count: **71** (full extraction across 1992-2025)
- 100% `pct_amount` (constant $500,000 USD per laureate)
- `currencies = [USD]`
- 1 distinct `funder_scheme` ('Blue Planet Prize'), 1 distinct `funding_type` ('prize')
- 34 distinct years (1992-2025)
- 65 individuals + 6 organizations (verified via `lead_investigator.given_name IS NULL` ↔ organization)
- `declined = FALSE` for all rows

In [ ]:
%sql
SELECT COUNT(*) AS total_blue_planet_award_rows FROM openalex.awards.blue_planet_prize_awards;

In [ ]:
%sql
DESCRIBE openalex.awards.blue_planet_prize_awards;

In [ ]:
%sql
-- §6.3 Data completeness
SELECT
    COUNT(*) AS total,
    COUNT(display_name) AS has_title,
    COUNT(description) AS has_description,
    COUNT(amount) AS has_amount,
    COUNT(start_date) AS has_start_date,
    COUNT(lead_investigator) AS has_pi,
    ROUND(COUNT(display_name) * 100.0 / COUNT(*), 1) AS pct_title,
    ROUND(COUNT(start_date) * 100.0 / COUNT(*), 1) AS pct_dates,
    ROUND(COUNT(amount) * 100.0 / COUNT(*), 1) AS pct_amount,
    ROUND(COUNT(description) * 100.0 / COUNT(*), 1) AS pct_description
FROM openalex.awards.blue_planet_prize_awards;

In [ ]:
%sql
-- §6.7 amount + currency fail-fast (NOT waived).
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS has_amount,
    ROUND(COUNT(amount) * 100.0 / COUNT(*), 1) AS pct_amount,
    COUNT(DISTINCT currency) AS distinct_currencies,
    collect_set(currency) AS currencies,
    ROUND(MIN(amount), 0) AS min_amount,
    ROUND(MAX(amount), 0) AS max_amount,
    ROUND(AVG(amount), 0) AS avg_amount
FROM openalex.awards.blue_planet_prize_awards;

In [ ]:
%sql
-- Year distribution + individuals/orgs mix per year
SELECT start_year,
       COUNT(*) AS rows,
       SUM(CASE WHEN lead_investigator.given_name IS NULL AND lead_investigator IS NOT NULL THEN 1 ELSE 0 END) AS orgs,
       SUM(CASE WHEN lead_investigator.given_name IS NOT NULL THEN 1 ELSE 0 END) AS individuals,
       ROUND(SUM(amount)/1e6, 2) AS total_usd_millions
FROM openalex.awards.blue_planet_prize_awards
WHERE start_year IS NOT NULL
GROUP BY start_year
ORDER BY start_year ASC;

In [ ]:
%sql
-- Sample notable laureates (most recent + earliest)
SELECT id, SUBSTRING(display_name, 1, 80) AS title,
       lead_investigator.given_name AS given,
       lead_investigator.family_name AS family,
       SUBSTRING(lead_investigator.affiliation.name, 1, 60) AS affiliation,
       amount, start_year
FROM openalex.awards.blue_planet_prize_awards
WHERE start_year >= 2023 OR start_year <= 1994
ORDER BY start_year DESC, family
LIMIT 15;

In [ ]:
%sql
-- Organizations vs individuals split — confirms 6/65 ratio.
SELECT
    CASE WHEN lead_investigator.given_name IS NULL THEN 'organization' ELSE 'individual' END AS recipient_kind,
    COUNT(*) AS rows
FROM openalex.awards.blue_planet_prize_awards
GROUP BY recipient_kind;

In [ ]:
%sql
-- Funder struct populated correctly?
SELECT funder.id, funder.display_name, funder.ror_id, funder.doi, COUNT(*) AS rows
FROM openalex.awards.blue_planet_prize_awards
GROUP BY funder.id, funder.display_name, funder.ror_id, funder.doi;

In [ ]:
%sql
-- Declined boolean must be FALSE everywhere — Foundation doesn't publish
-- declined nominees. (Schema parity with PRIZE PATTERN.)
SELECT declined, COUNT(*) AS rows
FROM openalex.awards.blue_planet_prize_awards
GROUP BY declined;